# Process Reward Model (PRM) — Interactive Demo

This notebook demonstrates a **rubric-based Process Reward Model** that scores individual reasoning steps in math problem-solving. Unlike an Outcome Reward Model (ORM) that only evaluates the final answer, a PRM audits *how* the model reasons at each step.

This is Phase 1 of alignment research: start with an objective domain (math), validate the rubric architecture, then extend to subjective alignment criteria.

## 1. Setup

Make sure you have Ollama installed and running with the `phi3:mini` model:

```bash
pip install -r requirements.txt
ollama pull phi3:mini
```

In [1]:
import sys
sys.path.insert(0, "..")

from prm.rubric import MATH_RUBRIC, ALIGNMENT_RUBRIC, StepScore
from prm.splitter import split_reasoning_chain
from prm.scorer import OllamaScorer
from prm.data import generate_curated_dataset, save_dataset
from prm.visualize import render_step_heatmap, render_comparison, render_aggregate

import matplotlib.pyplot as plt
%matplotlib inline

print("Imports OK")

Imports OK


In [2]:
scorer = OllamaScorer(model_name="phi3:mini", rubric=MATH_RUBRIC)

if scorer.verify_connection():
    print("Ollama connection verified — phi3:mini is available")
else:
    print("WARNING: Cannot reach Ollama or phi3:mini not found.")
    print("Run: ollama pull phi3:mini")

Ollama connection verified — phi3:mini is available


## 2. What is a Process Reward Model?

### Outcome Reward Model (ORM)
An ORM scores only the **final output**. It answers: *"Is this answer correct?"* This is a binary signal — you know the answer is wrong, but not *where* or *why* the reasoning failed.

### Process Reward Model (PRM)
A PRM scores **each intermediate step**. It answers: *"Is step 3 logically sound? Does step 5 contain a calculation error?"* This gives a fine-grained signal that is:
- **Interpretable** — you can see exactly where reasoning breaks down
- **Auditable** — each step gets an independent score
- **Trainable** — you can reinforce or penalize specific reasoning patterns

### Why this matters for alignment
In alignment, the *process* matters as much as the *outcome*. A model that reaches a "safe" answer via deceptive reasoning is dangerous. A PRM can audit the reasoning chain for honesty, transparency, and logical integrity — not just whether the final output looks harmless.

```
ORM:  Problem → [entire reasoning] → Score (0.8)
                                       ↑ tells you nothing about steps

PRM:  Problem → Step 1 (0.95) → Step 2 (0.90) → Step 3 (0.30) → Step 4 (0.85)
                                                   ↑ the error is HERE
```

## 3. Live Scoring Demo

Let's score a correct math reasoning chain step by step.

In [3]:
problem = "A store sells notebooks for $4 each. If Maria buys 7 notebooks and pays with a $50 bill, how much change does she receive?"

correct_chain = """Step 1: The cost of one notebook is $4.
Step 2: Maria buys 7 notebooks, so the total cost is 7 × $4 = $28.
Step 3: Maria pays with a $50 bill.
Step 4: Her change is $50 − $28 = $22."""

print("Problem:", problem)
print("\nReasoning chain:")
print(correct_chain)

Problem: A store sells notebooks for $4 each. If Maria buys 7 notebooks and pays with a $50 bill, how much change does she receive?

Reasoning chain:
Step 1: The cost of one notebook is $4.
Step 2: Maria buys 7 notebooks, so the total cost is 7 × $4 = $28.
Step 3: Maria pays with a $50 bill.
Step 4: Her change is $50 − $28 = $22.


In [ ]:
print("Scoring each step... (this calls phi3:mini via Ollama for each step)\n")

correct_scores = scorer.score_chain(problem, correct_chain)

for s in correct_scores:
    status = "✓" if s.is_complete() else "⚠"
    print(f"{status} Step {s.step_index + 1}: {s.step_text[:60]}...")
    for criterion, score in s.scores.items():
        bar = "█" * int((score or 0) * 20) + "░" * (20 - int((score or 0) * 20))
        print(f"    {criterion:20s} [{bar}] {score:.2f}" if score is not None else f"    {criterion:20s} [FAILED]")
    print()

Scoring each step... (this calls phi3:mini via Ollama for each step)



In [ ]:
fig = render_step_heatmap(correct_scores, title="Correct Reasoning Chain — Rubric Scores")
plt.show()

## 4. Error Detection

Now let's introduce a subtle arithmetic error and see if the PRM catches it.

In [ ]:
corrupted_chain = """Step 1: The cost of one notebook is $4.
Step 2: Maria buys 7 notebooks, so the total cost is 7 × $4 = $32.
Step 3: Maria pays with a $50 bill.
Step 4: Her change is $50 − $32 = $18."""

print("Corrupted chain (error in Step 2: 7×4 should be 28, not 32):")
print(corrupted_chain)
print("\nScoring...\n")

corrupted_scores = scorer.score_chain(problem, corrupted_chain)

for s in corrupted_scores:
    status = "✓" if s.mean_score() and s.mean_score() > 0.7 else "✗"
    mean = s.mean_score()
    print(f"{status} Step {s.step_index + 1} (mean: {mean:.2f}): {s.step_text[:60]}..." if mean else f"? Step {s.step_index + 1}: {s.step_text[:60]}")

In [ ]:
fig = render_comparison(
    correct_scores, corrupted_scores,
    correct_title="Correct (7×4=28)",
    incorrect_title="Corrupted (7×4=32)",
)
plt.show()

## 5. Rubric Exploration

The rubric is the key abstraction. Let's inspect what our math rubric looks like and how it translates into the scoring prompt.

In [ ]:
print("=" * 60)
print(f"Rubric: {MATH_RUBRIC.name}")
print(f"Description: {MATH_RUBRIC.description}")
print("=" * 60)

for c in MATH_RUBRIC.criteria:
    print(f"\n  Criterion: {c.name}")
    print(f"  Domain:    {c.domain}")
    print(f"  Range:     {c.score_range}")
    print(f"  Desc:      {c.description}")

print("\n" + "=" * 60)
print("Prompt fragment sent to the model:")
print("=" * 60)
print(MATH_RUBRIC.to_prompt_fragment())

In [ ]:
fig = render_aggregate(correct_scores, title="Correct Chain — Mean Scores")
plt.show()

fig = render_aggregate(corrupted_scores, title="Corrupted Chain — Mean Scores")
plt.show()

## 6. Dataset Generation

Generate a curated dataset of correct and corrupted reasoning chains for future fine-tuning.

In [ ]:
dataset = generate_curated_dataset()

n_correct = sum(1 for ex in dataset if ex.is_correct)
n_corrupted = sum(1 for ex in dataset if not ex.is_correct)
corruption_types = set(ex.corruption_type for ex in dataset if ex.corruption_type)

print(f"Generated {len(dataset)} examples:")
print(f"  Correct:   {n_correct}")
print(f"  Corrupted: {n_corrupted}")
print(f"  Corruption types: {corruption_types}")

In [ ]:
print("\n--- Example: Correct chain ---")
ex = [e for e in dataset if e.is_correct][0]
print(f"Problem: {ex.problem}")
print(f"Answer: {ex.answer}")
print(f"Chain:\n{ex.chain}")

print("\n--- Example: Corrupted chain ---")
ex = [e for e in dataset if not e.is_correct][0]
print(f"Problem: {ex.problem}")
print(f"Corruption: {ex.corruption_type}")
print(f"Chain:\n{ex.chain}")

In [ ]:
path = save_dataset(dataset)
print(f"Dataset saved to: {path}")

## 7. Alignment Bridge Preview

The same PRM architecture extends to alignment by swapping the rubric. Here's how the math criteria map to alignment criteria:

| Math Criterion | Alignment Criterion | What changes |
|---|---|---|
| `correctness` | `factual_accuracy` | Evaluates factual claims instead of computation |
| `logical_coherence` | `reasoning_transparency` | Checks for transparent, non-deceptive reasoning |
| *(new)* | `honesty` | Is the model faithfully representing its reasoning? |
| *(new)* | `harmlessness` | Is the step free from harmful content? |

The key insight: **the architecture doesn't change, only the rubric does.** The scorer, splitter, visualizer, and training pipeline all work identically — you just define different criteria.

In [ ]:
print("MATH RUBRIC")
print("=" * 40)
for c in MATH_RUBRIC.criteria:
    print(f"  {c.name:25s} (domain: {c.domain})")

print("\nALIGNMENT RUBRIC")
print("=" * 40)
for c in ALIGNMENT_RUBRIC.criteria:
    print(f"  {c.name:25s} (domain: {c.domain})")

print("\nSame OllamaScorer class, different rubric:")
print("  math_scorer = OllamaScorer(rubric=MATH_RUBRIC)")
print("  alignment_scorer = OllamaScorer(rubric=ALIGNMENT_RUBRIC)")

## Next Steps

1. **Phase 2: LoRA Fine-Tuning** — Use the generated dataset (with human label corrections) to train a LoRA adapter on Phi-3-mini via `mlx-lm`. This replaces prompt-based scoring with faster, more consistent inference.

2. **Alignment Extension** — Swap to the alignment rubric, curate a dataset of reasoning chains with alignment-relevant labels, and fine-tune a second adapter.

3. **Evaluation** — Compare PRM step-level scores against human judgments to measure calibration and inter-annotator agreement.